<center> MULTIPROCESSING IN PYTHON

In [ ]:
import multiprocessing, os

from multiprocessing import Process
print(f'__name__:{__name__}')

def run_function(x):
    print(f'[run function] process ID: {os.getpid()}')
    print(f'valore di x: {x}')

multiprocessing.set_start_method("spawn")
p1 = multiprocessing.Process(target=run_function, args=(20,))
p1.start()
p1.join()

In [ ]:
import multiprocessing, os

from multiprocessing import Process

print(f'PID: {os.getpid()}, __name__:{__name__}')

def run_function(x):
    print(f'[run function] process ID: {os.getpid()}')
    print(f'valore di x: {x}')

if __name__ == "__main__":
    multiprocessing.set_start_method("spawn")
    p1 = multiprocessing.Process(target=run_function, args=(20,))
    p1.start()
    p1.join()

SU LINUX (MIA VM):
1) Se non faccio nulla -> va di default fork, funziona ma è unsafe | Stampa singola
2) Se specifico solo spawn -> _context has already been set_, doppia stampa
3) Se specifico spawn -> va con doppia stampa [idem forkserver]

Classe Multiprocess.Process -> Metodi analoghi a threading.Thread

Ho quindi anche qui 2 modi per creare un nuovo processo

In [ ]:
#metodo 1, istanziare un oggetto Process passando un "callable object"
import multiprocess as mp

def func():
    print ('Process running')
    return

if __name__ == '__main__':

    # creating process
    p = mp.Process(target = func)
    # starting process
    p.start()
    # wait until the process finishes
    p.join()

In [ ]:
#metodo 2, creare una classe che estende quella Process, ridefinendo il metodo run()
import multiprocess as mp

class MyProcess(mp.Process):

    def run(self):
        print ('Process running')
        return

if __name__ == '__main__':

    # creating process
    p = MyProcess()
    # starting process
    p.start()
    # wait until the process finishes
    p.join()

Anche per i Process si possono usare primitive di sincronizzazione equivalenti a quelle di threading

In [ ]:
from multiprocess import Process, Lock

def f(l, i):
    l.acquire()
    try:
        print('hello world', i)
    finally:
        l.release()

if __name__ == '__main__':
    lock = Lock()
    for num in range(10):
        Process(target=f, args=(lock, num)).start()

hello world 0
hello world 1
hello world 2


hello world 3
hello world 4
hello world 5
hello world 6
hello world 7
hello world 8
hello world 9


_______________________________________________________________________________________________________________________________

PIPE -> si invia usando _send()_ e _receive()_

In [2]:
from multiprocessing import Process, Pipe

def parentData(parent): #run function parent
    """
    Invia Hello sulla Pipe
    """
    parent.send(['HELLO'])
    parent.close()
    
def childData(child): #run function figlio
    """
    Invia Bye sulle Pipe
    """
    child.send(['BYE'])
    child.close()
    
if __name__ == '__main__':
    
    parent_conn, child_conn = Pipe() #creo la Pipe
    
    p1 = Process(target=parentData, args=(parent_conn, ))
    p1.start()
    
    p2 = Process(target=childData, args= (child_conn,))
    p2.start()
    
    print("Su parent_conn ricevo: ", parent_conn.recv())
    print("Su child_conn ricevo: ",child_conn.recv())
    
    p1.join()
    p2.join()

Su parent_conn ricevo:  ['BYE']
Su child_conn ricevo:  ['HELLO']


_____________________________________________________________________

QUEUE -> comunico con put(), put_nowait(), get, get_nowait()

In [6]:
from multiprocess import Process, Queue

def f(q:Queue):
    """
    Aggiunge la tupla alla coda, bloccante e senza timeout(default)
    """
    q.put([42, None, 'hello'])
    
if __name__ == '__main__':
    q=Queue()
    
    p=Process(target=f, args=(q,))
    
    p.start()
    
    print('Coda ha ricevuto', q.get())
    #print('Coda ha ricevuto', q.get()) #se ne metto uan seconda resta in attesa
    
    p.join()

Coda ha ricevuto [42, None, 'hello']


____________________________________________________________________

SHARED MEMORY -> Sfrutta process- e thread safe Value e Array

In [13]:
from multiprocess import Process, Value, Array

def f(n,a):
    """
    Assegna valore a Value e inverte di segno l'array
    """
    n.value = 3.1415927
    
    for i in range(len(a)):
        a[i] = - a[i]
        
if __name__ == "__main__":
    
    num = Value('d', 0.0) #sto specificando tipo di pggetto ritornato e valore
    arr = Array('i', range(10))
    
    p = Process(target=f, args=(num, arr))
    p.start()
    
    p.join()
    
    print(num.value)
    print(arr[:])

3.1415927
[0, -1, -2, -3, -4, -5, -6, -7, -8, -9]
